# Parallelization

There are two main ways to parallelize a computation:

- Use multiple computers (or better - multiple CPUs).

- Use multiple threads on the same CPU. 

The two approaches are not mutually exclusive, and can be combined to achieve even greater performance. However, they require different programming models and tools.

# Multi-threading

Multi-threading aims at exploiting the full potential of a single CPU.
The point is that normal applications do not use the CPU to its full capacity, and there are often idle cycles that can be used to perform additional computations.
By using multiple threads, we can keep the CPU busy and achieve better performance.
The CPU must support multi-threading, but this is a common feature in modern processors.
The main challenge is to design the program in such a way that it is split into independent tasks that can be executed in parallel without interfering with each other.
This often requires careful consideration of data dependencies and synchronization between threads.
We will not discuss multi-threading in this course, but you can find the basics in Pavel Stransky's [lecture notes](https://raw.githubusercontent.com/PavelStransky/PCInPhysics/main/Poznamky.pdf).

# Multi-processing

Using multiple CPUs is useful when it is possible to replace a long-running program with several shorter runs of the same program.
Typical situations include:

- MC integration, where the same program is run multiple times (just with different random seeds) to obtain better statistics.

- Big data processing, where the data can be split into smaller chunks that can be processed independently.

In these cases, we can use a cluster of computers to run multiple instances of the program in parallel, and then combine the results at the end.

# Terminology

- **Node**: The whole server in a cluster.

- **CPU**: Central Processing Unit, the processing chip on a node.

- **Core**: A single processing unit within a CPU. Modern CPUs have multiple cores, which can execute instructions independently. Basically, one core is needed for each process you run.

# Chimera cluster

The Chimera cluster is a high-performance computing cluster that is available to students and researchers at the Faculty of Mathematics and Physics at Charles University.
It is the biggest part of the Faculty Metacenter (often called just HPC for High Performance Computing), which also includes other smaller clusters and a GPU cluster.

Detailed information about the HPC cluster can be found on the [HPC GitLab page](https://gitlab.mff.cuni.cz/mff/hpc/clusters) or its [web page](https://www.mff.cuni.cz/en/hpc-cluster/general-information).
A great [practical guide](https://ipnp.cz/?page_id=8244) on how to use the Chimera cluster was written by Daniel Scheirich for the needs of IPNP (Institute of Particle and Nuclear Physics) researchers and students, but it is a good starting point for everyone.

# SSH

Standard tool for connecting to a remote server is SSH (Secure Shell).
It allows you to log in to the server and run commands as if you were sitting in front of it.
You can also use SSH to transfer files between your local machine and the server.
To connect to the Chimera cluster, you need to use the following command in your PowerShell:
```powershell
ssh <username>@hpc.troja.mff.cuni.cz
```
Replace `<username>` with your actual username on the cluster (the same as for the SIS system).
You will be prompted to enter your password, and then you will be logged in to the cluster.

# SSH config file

To avoid typing `<username>@hpc.troja.mff.cuni.cz` every time, you can create an SSH config file that defines a shortcut for the connection. This way, you can simply type e.g. `ssh hpc` to connect to the cluster. Just create a folder `C:\Users\<your_windows_username>\.ssh\` and inside it create a file `config` with the following content:
```
Host hpc
    HostName hpc.troja.mff.cuni.cz
    User <username>
```
Replace `<username>` with your actual username on the cluster. After that, you can connect to the cluster by simply typing `ssh hpc` in your PowerShell.

# SSH key-based authentication

If you do not want to enter your password every time you connect to the cluster, you can set up SSH key-based authentication. This involves generating a pair of SSH keys (a private key and a public key) on your local machine, and then copying the public key to the cluster. This way, you can authenticate using your private key instead of a password.

To generate an SSH key pair, you can use the following command in your PowerShell:
```powershell
ssh-keygen -t ed25519 -C "<your_email@example.com>"
```
This will create a private key (usually named `id_ed25519`) and a public key (usually named `id_ed25519.pub`) in the `C:\Users\<your_windows_username>\.ssh\` directory.
When prompted for a file to save the key, you can press Enter to accept the default location. You can also choose to set a passphrase for added security, or leave it empty for no passphrase.
Use the password to protect your private key!

Next, you need to copy the public key to the cluster. You can do this using the `ssh-copy-id` command:
```powershell
type $HOME\.ssh\id_ed25519.pub | ssh hpc "mkdir -p .ssh && tee .ssh/authorized_keys"
```
You could also use `<username>@hpc.troja.mff.cuni.cz` instead of `hpc` where `<username>` is your actual username on the cluster.
Another way to do the same thing is to copy the whole content of `id_ed25519.pub` from the local machine to the end of the `~/.ssh/authorized_keys` file on the remote machine.

# SSH agent to keep your private key secure

We now moved from the necessity to type in your hpc password every time you log in to Chimera to typing your private key's password every time you log in.
This is not very efficient :)
Fortunately, there exists a tool keeping your private key ready but secure for as long as your local session runs.
You need to use the SSH Agent Service on Windows (or something similar on other platforms).
To start the agent, do:

1. Open PowerShell as Administrator (Right-click Start > Terminal (Admin)).

2. Run the following commands to start the service and set it to automatic:
```powershell
Set-Service -Name ssh-agent -StartupType Automatic
Start-Service ssh-agent
```

If you wanted the agent to remember your key, do:
```powershell
ssh-add $env:USERPROFILE\.ssh\id_ed25519
```


# Edit files on the cluster using VS Code

Install the [Remote - SSH extension](https://marketplace.visualstudio.com/items?itemName=ms-vscode-remote.remote-ssh) to edit files on the cluster using VS Code. After installing the extension, you can connect to the cluster by clicking on the icon with two arrows in the bottom left corner of VS Code and selecting "Connect to Host". Then, select the host you defined in your SSH config file (e.g. `hpc`), and VS Code will establish a connection to the cluster. You can then open files and folders on the cluster directly from VS Code, and edit them as if they were on your local machine.

> IMPORTANT: Do not use the terminal in VS Code to run your programs on the cluster! The point is that the terminal session is open on the login node, which is not meant for running programs - it is shared by all users and has limited resources. Read the text below for more information about how to run programs on the cluster.

# Slurm

User jobs on the Chimera cluster are managed by a job scheduler called Slurm. It is responsible for allocating resources (CPUs, memory, etc.) to user jobs, and for scheduling the execution of jobs on the cluster. To run a program on the cluster, you need to submit a job to Slurm. Slurm will then schedule your job to run on the cluster when the required resources are available. There are two basic types of jobs: interactive jobs and batch jobs.

# Interactive jobs

These are jobs that are run in an interactive session on the cluster. When using JupyterHub, you are running an interactive session on the cluster. Another way to run an interactive session is to use the `salloc` command in the terminal on the cluster login node. This will allocate the requested resources and give you a shell on a computer node where you can run your program interactively. An example command to start an interactive session with 1 core and 1 GB of memory for 24 hours in the `ffa` partition:
```bash
salloc -p ffa --cpus-per-task 1 --mem 1G --time=24:00:00
```

Note that you might need to wait for the resources to become available in the `ffa` partition. A high priority partition is `edu`; unfortunately, it has a short time limit (4 hours):
```bash
salloc -p edu --cpus-per-task 1 --mem 1G --time=4:00:00
```

# Batch jobs

Suitable for programs that do not require user interaction, while interactive jobs are useful for debugging and testing your code before submitting it as a batch job.
The heavy lifting on the cluster is done by batch jobs.
You need to use the `sbatch` command in combination with a job script. The job script contains the commands to run your program, as well as some Slurm directives that specify the resources required for the job. An example job script that runs just a simple echo command, using 1 core and 100 MB of memory for 1 minute in the `ffa-preempt` partition:
```bash
#!/bin/bash
#SBATCH -p ffa-preempt
#SBATCH --cpus-per-task 1
#SBATCH --mem 100M
#SBATCH --time 00:01:00
#SBATCH --job-name test

# Here, do the real work
# Just a simple example:
echo "Hello from the cluster!"
```

The first line `#!/bin/bash` is important, as it tells the system that this is a bash script. The lines starting with `#SBATCH` are Slurm directives that specify the resources required for the job. The rest of the script contains the commands to run your program.

> Tip: Create a folder `test` on the cluster and put the script there. Name it whatever you like, e.g., `job.sh`. Then, submit the job using the command `sbatch job.sh` while being in the `test` folder. This way, the log files will be created in the same folder, and you can easily check them after the job is finished. The textual output of the job will be in a file named `slurm-<job_id>.out`, where `<job_id>` is the ID assigned to your job by Slurm.

# Slurm directives

- `#SBATCH -p <partition>` - specifies the partition to run the job in. The `ffa` partition is a good choice for most jobs, but it might have a long queue. The `edu` partition has a very short time limit (4 hours) and just one job per user, but it has a higher priority. The `ffa-preempt` partition is a good choice for short jobs that can be preempted by other jobs.

- `#SBATCH --cpus-per-task <num>` - specifies the number of CPU cores required for the job.

- `#SBATCH --mem <amount>` - specifies the amount of memory required for the job.

- `#SBATCH --time <time>` - specifies the maximum time for the job to run. The format is `DD-HH:MM:SS`.

- `#SBATCH --job-name <name>` - specifies a name for the job, which can be useful for identifying it in the job queue.

- `#SBATCH --output <file>` - specifies the file to which the standard output of the job will be written. By default, it is `slurm-<job_id>.out`.

- `#SBATCH --error <file>` - specifies the file to which the standard error of the job will be written.

# Slurm commands

- `squeue` - shows the list of currently running and pending jobs. You can use the `--me` option to show only your jobs.

- `scancel` - cancels a running or pending job. You can specify the job ID or use the `-u <username>` option to cancel all your jobs.

- `sinfo` - shows information about the cluster, including the available partitions and their status.

# Job arrays

If you need to run the same program multiple times with different parameters (e.g., different random seeds), you can use job arrays. A job array allows you to submit multiple jobs with a single command, and each job in the array will have a unique index that can be used to specify different parameters for each job.
To create a job array, use the `--array` option. The unique index of each job in the array can be accessed using the environment variable `SLURM_ARRAY_TASK_ID` (and the `%a` placeholder in the output and error file names).
Example job script:
```bash
#!/bin/bash
#SBATCH --array 1-10
#SBATCH -p ffa-preempt
#SBATCH --cpus-per-task 1
#SBATCH --mem 100M
#SBATCH --time 00:01:00
#SBATCH --job-name test
#SBATCH --output slurm-%a.out
#SBATCH --error slurm-%a.err

# Here, do the real work
# Just a simple example:
echo "Hello from the cluster! Task ID: $SLURM_ARRAY_TASK_ID"
```

# Running a Python program in a batch job

For testing, create a simple Python script `generate.py`. The script should take the index of a job in the array as a command-line argument, and generate some random numbers, using the index to set the random seed. For example:
```python
import numpy as np
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("-i", "--index", type = int)
args = parser.parse_args()

np.random.seed(args.index)
a = np.random.normal(size = 1000000)

print(f"Generated {len(a)} random numbers with seed {args.index}")
print(f"Mean: {np.mean(a)}, Std: {np.std(a)}")
```

The script needs `numpy` to run, so make sure to use an appropriate Python environment on the cluster. The following venv serves our needs:
```bash
/singularity/ucjf/venv_praktikum/bin/activate
```

> IMPORTANT: When creating your own venv, make sure you are in an interactive session and not on the login node!

Finally, create a job script `job.sh` that runs the Python script in a job array:
```bash
#!/bin/bash
#SBATCH --array 1-10
#SBATCH -p ffa-preempt
#SBATCH --cpus-per-task 1
#SBATCH --mem 1G
#SBATCH --time 00:01:00
#SBATCH --job-name test
#SBATCH --output slurm-%a.out
#SBATCH --error slurm-%a.err

# Activate the Python virtual environment
source /singularity/ucjf/venv_praktikum/bin/activate

# Run the Python script with the index of the job in the array as an argument
python generate.py -i $SLURM_ARRAY_TASK_ID
```

Submit the job using the command `sbatch job.sh`.

# Generating the submission files from a Python script

Often, you need to run a large number of jobs with different parameters, and it can be tedious to create a separate job script for each job. In such cases, it can be useful to generate the job scripts from a Python script. This way, you can easily create a large number of job scripts with different parameters, and then submit them all at once. Often, you will also need to create the Python main script (which will be run by the job scripts) from a Python script.